# 1.5 — Register control: the same "casual, conversational tone" clause on the generic prompt and on every component

**Why.** In 1.4 half the residual (0.46 nats/response) looked like *register* variation (length, terseness,
hedging), and several of the weighted elicited components (`e58`, `e59`, `e35`) are broad enough that their
weights partly measure register rather than values. If we pin the register with one shared sentence, added
identically to $P_0$ and to all components, then (i) the residual should fall if it was register, and (ii)
the weights should move toward *behavioural* components. Everything else (framing `unknown`, questions,
seed, components, meta filter) is unchanged.

Data: `results/phase1/base_unknown_casual_v1` (sampled and scored with `--register casual`, then rescored
under the 80 elicited components with the same clause) vs. `base_unknown_v1` from 1.2/1.4.

In [ ]:
import os, sys, json, glob, textwrap, collections
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

REPO = Path.cwd().resolve().parent
sys.path.insert(0, str(REPO / "src"))
from persona_selection.mixture import em_weights, fit_and_evaluate, bootstrap_over_groups, greedy_forward_selection, is_meta_response

HAND = ["hhh", "fred", "evil", "sycophant", "formal", "neutral"]
desc = {m["name"]: m["body"] for m in json.load(open(REPO / "data" / "prompts" / "personas_elicited" / "_meta.json"))["kept"]}

def load(run):
    d = REPO / "results" / "phase1" / run
    rows = [json.loads(l) for l in open(d / "rows_ext.jsonl")] if (d / "rows_ext.jsonl").exists() else [json.loads(l) for l in open(d / "rows.jsonl")]
    ll = {i: dict(r["ll"]) for i, r in enumerate(rows)}
    for fn in sorted(glob.glob(str(d / "rows_elicited*.jsonl"))):
        for i, l in enumerate(open(fn)):
            ll[i].update(json.loads(l)["ll"])
    el = sorted({k for x in ll.values() for k in x if k.startswith("e") and k[1:].isdigit()})
    names = [n for n in HAND if n in ll[0]] + el
    idx = [i for i in range(len(rows)) if all(n in ll[i] for n in names) and not is_meta_response(rows[i]["response"])]
    return {"rows": [rows[i] for i in idx], "L": np.array([[ll[i][n] for n in names] for i in idx]), "l0": np.array([rows[i]["ll_generic"] for i in idx]),
            "nt": np.array([rows[i]["n_tokens"] for i in idx]), "groups": np.array([rows[i]["qidx"] for i in idx]), "names": names,
            "n_meta": len(rows) - len(idx), "cfg": json.loads((d / "config.json").read_text())}

D = {"plain": load("base_unknown_v1"), "casual": load("base_unknown_casual_v1")}
for k, d in D.items():
    print(f"[{k}] {len(d['rows'])} clean responses ({d['n_meta']} meta dropped) | {len(d['names'])} components | median length {np.median(d['nt']):.0f} tokens")
print("\ncasual generic prompt:\n" + D["casual"]["cfg"]["generic_prompt_example"])

## Did the clause change the samples?

Length, terseness, and a few register markers, before vs. after. If nothing moves here, the clause did
nothing and the rest of the notebook is moot.

In [ ]:
import re
def markers(rows):
    t = [r["response"].strip() for r in rows]
    return {"median tokens": np.median([r["n_tokens"] for r in rows]), "≤5 tokens": np.mean([r["n_tokens"] <= 5 for r in rows]),
            "starts with list/markdown": np.mean([bool(re.match(r"^(\d+\.|[-*]|\*\*)", x)) for x in t]),
            "contractions (I'm/you're/don't)": np.mean([bool(re.search(r"\b(I'm|you're|don't|can't|it's|that's)\b", x, re.I)) for x in t]),
            "hedge ('it depends'/'consider')": np.mean([bool(re.search(r"\b(it depends|consider|important to)\b", x, re.I)) for x in t]),
            "exclamation": np.mean(["!" in x for x in t]), "second person 'you'": np.mean([bool(re.search(r"\byou\b", x, re.I)) for x in t])}
M = {k: markers(d["rows"]) for k, d in D.items()}
print(f"{'marker':>34} {'plain':>8} {'casual':>8}")
for m in M["plain"]:
    print(f"{m:>34} {M['plain'][m]:8.2f} {M['casual'][m]:8.2f}")
print("\nrandom casual samples:")
rng = np.random.default_rng(0)
for i in rng.choice(len(D["casual"]["rows"]), 6, replace=False):
    r = D["casual"]["rows"][i]; print(f"  Q: {r['question'][:50]:50} | {textwrap.shorten(r['response'].strip(), 130)}")

## Fits: hand-written 6, then the full basis, plain vs. casual

In [ ]:
def fit(d, cols, n_boot=100):
    L = d["L"][:, cols]; r = fit_and_evaluate(L, d["l0"], d["groups"], d["nt"]); w, info = em_weights(L)
    b = bootstrap_over_groups(L, d["l0"], d["groups"], n_boot=n_boot, n_tokens=d["nt"])
    return {"w": w, "sd": b["w"].std(0), "kl": r["heldout"]["kl_per_response"], "kl_se": r["heldout"]["kl_se"], "gamma": info["gamma"]}

R = {}
for k, d in D.items():
    hand_cols = [d["names"].index(n) for n in HAND if n in d["names"]]
    R[(k, "hand6")] = fit(d, hand_cols); R[(k, "all")] = fit(d, list(range(len(d["names"]))))
    print(f"[{k}] held-out KL: hand-written 6 = {R[(k,'hand6')]['kl']:+.3f} ± {R[(k,'hand6')]['kl_se']:.3f} | all {len(d['names'])} = {R[(k,'all')]['kl']:+.3f} ± {R[(k,'all')]['kl_se']:.3f}")
print("\nhand-written weights (plain -> casual):")
for j, n in enumerate([n for n in HAND if n in D["plain"]["names"]]):
    print(f"  {n:>10}: {R[('plain','hand6')]['w'][j]:.3f} ± {R[('plain','hand6')]['sd'][j]:.3f}  ->  {R[('casual','hand6')]['w'][j]:.3f} ± {R[('casual','hand6')]['sd'][j]:.3f}")

In [ ]:
print("full-basis weights, components with ≥ 2% in either condition (plain -> casual):")
names = D["plain"]["names"]; assert names == D["casual"]["names"], "component sets differ"
wp, wc = R[("plain", "all")]["w"], R[("casual", "all")]["w"]
order = np.argsort(-(wp + wc))
print(f"{'component':>10} {'plain':>7} {'casual':>7}   description")
for j in order:
    if max(wp[j], wc[j]) < 0.02: continue
    print(f"{names[j]:>10} {wp[j]:7.3f} {wc[j]:7.3f}   {textwrap.shorten(desc.get(names[j], '(hand-written)'), 100)}")
print(f"\nevil weight: plain {wp[names.index('evil')]:.3f} -> casual {wc[names.index('evil')]:.3f}")

In [ ]:
# greedy selection under casual, to compare the K curve with 1.4
dc = D["casual"]
steps = greedy_forward_selection(dc["L"], dc["l0"], dc["groups"], dc["names"], dc["nt"], max_k=10)
for s in steps:
    print(f"K={s['k']:2d}: +{s['added']:>9} -> held-out KL {s['kl_heldout']:+.3f}  {textwrap.shorten(desc.get(s['added'], '(hand-written)'), 80)}")

In [ ]:
# what the top casual components claim
g = R[("casual", "all")]["gamma"]
for j in order[:5]:
    n = names[j]
    if wc[j] < 0.03: continue
    print("=" * 110); print(f"{n} (casual w={wc[j]:.3f}): {textwrap.shorten(desc.get(n, '(hand-written)'), 200)}")
    for i in np.argsort(-g[:, j])[:4]:
        print(f"  γ={g[i, j]:.2f} | Q: {dc['rows'][i]['question'][:45]:45} | {textwrap.shorten(dc['rows'][i]['response'].strip(), 120)}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
show = [j for j in order if max(wp[j], wc[j]) >= 0.02]
x = np.arange(len(show))
ax.bar(x - 0.2, wp[show], 0.4, label=f"plain (KL all {R[('plain','all')]['kl']:.2f})", color="0.6")
ax.bar(x + 0.2, wc[show], 0.4, label=f"casual (KL all {R[('casual','all')]['kl']:.2f})", color="#2C6FB3")
ax.set_xticks(x); ax.set_xticklabels([names[j] for j in show], rotation=45, ha="right"); ax.set_ylabel("fitted weight")
ax.legend(frameon=False); ax.spines[["top", "right"]].set_visible(False); ax.set_title("Full-basis weights with and without the shared register clause")
plt.tight_layout(); fig.savefig(REPO / "results" / "phase1" / "1.5_register_control.png", dpi=150); plt.show()
json.dump({"markers": {k: {m: float(v) for m, v in M[k].items()} for k in M}, "kl": {f"{k}_{b}": {"kl": R[(k, b)]["kl"], "kl_se": R[(k, b)]["kl_se"]} for k, b in R},
           "w_all": {"plain": dict(zip(names, wp.tolist())), "casual": dict(zip(names, wc.tolist()))}, "greedy_casual": steps},
          open(REPO / "results" / "phase1" / "1.5_register_control.json", "w"), indent=2)
print("saved results/phase1/1.5_register_control.{png,json}")

## What to look for

- **Did the residual fall?** If the full-basis KL drops well below 0.46, the register clause removed
  variation that no component could explain; if it barely moves, the residual is not register.
- **Did the weights move toward behaviour?** Broad register-like components (`e58`, `e59`, `e35`) losing
  weight to value-laden ones (sarcastic/selfish, caring, analytical) would be the intended effect.
- **Evil, again.** With register pinned, a malicious component competes only on *what* is said, not how;
  whether it regains weight is the sharpest reading of "selfish samples = sarcasm tail" from 1.4.
- Sanity: the sample markers must actually change (more contractions, fewer lists). If they don't, one
  sentence was not enough to move the register and the comparison is uninformative.